<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/email-management-api/blob/main/qwen25_email_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Dependencies

In [1]:
!pip install -q pandas openpyxl

In [19]:
import pandas as pd
import json
import ast
import torch
from sklearn.model_selection import train_test_split

Get Excel Sheet

In [3]:
from google.colab import files
uploaded = files.upload()
excel_file = next(iter(uploaded.keys()))
print("Uploaded:", excel_file)

Saving manual_classification_updated.xlsx to manual_classification_updated.xlsx
Uploaded: manual_classification_updated.xlsx


Extract the data from sheet

In [4]:
df = pd.read_excel(excel_file)

required_columns = [
    "subject",
    "from",
    "body",
    "sender_type",
    "is_action_required",
    "labels_to_remove",
    "labels_to_add"
]

df = df[required_columns]

print("Total records:", len(df))

Total records: 1981


In [5]:
def parse_labels(value):

    if isinstance(value, list):
        return value

    if pd.isna(value) or str(value).strip() == "":
        return []

    return ast.literal_eval(value)


def convert_record(row):

    input_data = {
        "subject": row["subject"],
        "from": row["from"],
        "body": row["body"],
        "sender_type": row["sender_type"],
        "is_action_required": row["is_action_required"]
    }

    output_data = {
        "labels_to_remove": parse_labels(row["labels_to_remove"]),
        "labels_to_add": parse_labels(row["labels_to_add"])
    }

    return {
        "input": json.dumps(input_data, ensure_ascii=False),
        "output": json.dumps(output_data, ensure_ascii=False)
    }


dataset = [
    convert_record(row)
    for _, row in df.iterrows()
]

print("Conversion completed!")
print("Total records:", len(dataset))

Conversion completed!
Total records: 1981


Save records as json

In [6]:
with open(
    "converted_dataset.jsonl",
    "w",
    encoding="utf-8"
) as f:

    for record in dataset:
        f.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

print("JSONL file created successfully!")

JSONL file created successfully!


System Prompt

In [13]:

all_labels = set()

for column in ["labels_to_add", "labels_to_remove"]:
    for value in df[column]:
        all_labels.update(parse_labels(value))

ALLOWED_LABELS = sorted(all_labels)

print("Total unique labels:", len(ALLOWED_LABELS))

Total unique labels: 47


In [15]:
SYSTEM_PROMPT = f"""
You are an AI email management assistant.

Analyze the provided email and determine:

1. Which labels should be added.
2. Which existing labels should be removed.
3. Whether the email requires an action or reply from the user.

Allowed labels:
{json.dumps(ALLOWED_LABELS, ensure_ascii=False)}

Rules:
- Use only labels from the allowed labels list.
- An email can have multiple labels.
- Do not add unnecessary labels.
- Do not remove labels unnecessarily.
- Return only valid JSON.
- Do not include explanations or Markdown.

Required output format:
{{
    "labels_to_add": [],
    "labels_to_remove": [],
    "is_action_required": false
}}
"""

Conversation dataset

In [16]:
def create_conversation(row):

    input_data = {
        "from": str(row["from"]),
        "subject": str(row["subject"]),
        "body": str(row["body"]),
        "sender_type": str(row["sender_type"])
    }

    output_data = {
        "labels_to_add": parse_labels(row["labels_to_add"]),
        "labels_to_remove": parse_labels(row["labels_to_remove"]),
        "is_action_required": bool(row["is_action_required"])
    }

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": json.dumps(input_data, ensure_ascii=False)
        },
        {
            "role": "assistant",
            "content": json.dumps(output_data, ensure_ascii=False)
        }
    ]

    return {"messages": messages}


conversations = [
    create_conversation(row)
    for _, row in df.iterrows()
]

print("Total conversations:", len(conversations))

Total conversations: 1981


In [18]:
with open("email_conversations.jsonl", "w", encoding="utf-8") as f:

    for example in conversations:
        f.write(
            json.dumps(example, ensure_ascii=False) + "\n"
        )

print("Dataset saved successfully!")

Dataset saved successfully!


Split the dataset

In [20]:
train_data, temp_data = train_test_split(
    conversations,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

validation_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Training:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

Training: 1584
Validation: 198
Test: 199


In [21]:
def save_jsonl(data, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for record in data:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

save_jsonl(train_data, "train.jsonl")
save_jsonl(validation_data, "validation.jsonl")
save_jsonl(test_data, "test.jsonl")

print("All datasets saved successfully!")

All datasets saved successfully!


Check GPU Availability Status

In [7]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("No GPU detected. Please enable GPU in Colab.")

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU Memory: 14.56 GB
